<a href="https://colab.research.google.com/github/SunilJoon/Telecom-AI-Co-Pilot/blob/main/Telecom_AI_Applications.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers langchain-community
!pip install -q -U langchain-huggingface
print("\nStep 1 Complete: Packages Installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.

Step 1 Complete: Packages Installed


## Step 4: Provide 3GPP Content and Generate Q&A Pairs

Now, you need to provide the actual 3GPP content from which you want to generate the Q&A pairs. For this example, I'll use a placeholder text. **You should replace this with your actual 3GPP document excerpts or loaded content.**

In [2]:
import json
import re # Import regex module
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Initialize the Language Model (Moved from cell 7515502c)
model_name = "gpt2" # Changed to 'gpt2' as requested
generator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name,
    device=-1, # Changed to -1 to force CPU usage to resolve CUDA error
    return_full_text=False, # Crucial: only return generated text, not the prompt
    # Removed max_length to avoid conflict warnings and rely on max_new_tokens
    max_new_tokens=500 # Control the length of the *newly* generated response, set to be conservative
)
llm = HuggingFacePipeline(pipeline=generator)
print(f"\nLanguage model '{model_name}' initialized.")

# Define the prompt template (Moved from cell a45b5027)
qna_prompt_template = """You are an expert in 3GPP telecommunications. Your task is to read the following 3GPP specification text and generate exactly 10 question-answer pairs based on the content. The questions should be specific and the answers should be concise and directly derived from the provided text. Make sure the questions cover different aspects of the text.\n\nText: {text}\n\nGenerate 10 Q&A pairs in the following JSON format:\n[\n  {{\"question\": \"What is X?\", \"answer\": \"Y.\"}},\n  {{\"question\": \"How does A work?\", \"answer\": \"B.\"}}\n]\n"""

# Create the prompt template instance (Moved from cell a45b5027)
qna_prompt = PromptTemplate(
    input_variables=["text"],
    template=qna_prompt_template
)
print("Q&A generation prompt defined.")

# Placeholder for 3GPP content. REPLACE THIS WITH YOUR ACTUAL DATA.
# You might load this from a file, scrape a webpage, or have it as a string.
example_3gpp_text = """3GPP Technical Specification 23.501 V16.8.0 (2020-12)
5G System Architecture

5.1 General Principles
The 5G System architecture is defined to support various services and use cases, including enhanced Mobile Broadband (eMBB), Ultra-Reliable Low Latency Communication (URLLC), and massive Machine Type Communication (mMTC). It builds upon the existing 4G LTE architecture with significant enhancements to meet the requirements of new services. Key architectural principles include service-based architecture, network slicing, and control/user plane separation.

5.2 Network Functions
Key Network Functions (NFs) in the 5G System include: AMF (Access and Mobility Management Function), SMF (Session Management Function), UPF (User Plane Function), UDM (Unified Data Management), AUSF (Authentication Server Function), PCF (Policy Control Function), NEF (Network Exposure Function), and NRF (NF Repository Function).

5.2.1 AMF (Access and Mobility Management Function)
The AMF is responsible for Connection Management (CM) and Mobility Management (MM). It handles UE registration, connection setup, reachability management, and mobility between 3GPP access and non-3GPP access. The AMF also performs authentication and authorization with the AUSF and UDM.

5.2.2 SMF (Session Management Function)
The SMF is responsible for session management, including session establishment, modification, and release. It selects and controls the UPF, allocates IP addresses, and manages QoS flows.

5.2.3 UPF (User Plane Function)
The UPF is the core component for the user plane data handling. It performs packet routing and forwarding, inter-system mobility, and acts as an anchor point for session continuity.
"""

# Use the LLM to generate Q&A pairs
print("Generating Q&A pairs...")

# Format the prompt with the text
formatted_prompt = qna_prompt.format(text=example_3gpp_text)

# Generate the response using the invoke method
raw_response = llm.invoke(formatted_prompt)

# Attempt to parse the JSON output more robustly using regex
qna_pairs = []
# Regex to find all occurrences of {"question": "...", "answer": "..."}
# It's made more robust by handling potential escaped quotes within the string values
json_pattern = re.compile(r'\{\s*"question"\s*:\s*"(.*?)(?<!\\)"\s*,\s*"answer"\s*:\s*"(.*?)(?<!\\)"\s*\}')

matches = json_pattern.findall(raw_response)

if matches:
    for q_text, a_text in matches:
        # Unescape quotes that might have been escaped by the model within the strings
        question = q_text.replace('\\"', '"')
        answer = a_text.replace('\\"', '"')
        qna_pairs.append({"question": question, "answer": answer})

    if qna_pairs:
        print(f"\nSuccessfully extracted {len(qna_pairs)} potential Q&A pairs:")
        for i, qa in enumerate(qna_pairs):
            print(f"Q{i+1}: {qa['question']}")
            print(f"A{i+1}: {qa['answer']}\n")
    else:
        print("\nNo valid Q&A pairs could be extracted from the model output. Raw response:\n")
        print(raw_response)
else:
    print("\nNo JSON objects of the expected Q&A format were found in the response. Raw response:\n")
    print(raw_response)

print("\nStep 4 Complete: Q&A generation attempt finished. Review the output for quality and completeness.")

/tmp/ipykernel_569/2914130634.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_569/2914130634.py:18: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generator)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Language model 'gpt2' initialized.
Q&A generation prompt defined.
Generating Q&A pairs...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



No JSON objects of the expected Q&A format were found in the response. Raw response:


The Q&A pairs created by the AMF are then referenced and referenced in a JSON file.

5.2.4 AUSF (User Plane Function)

AUSF is responsible for session management, including session establishment, modification, and release. It selects and controls the UPF, allocates IP addresses, and manages QoS flows.

5.2.5 UDM (Unified Data Management Function)

UDM is responsible for data management, including data management, in the UDM. It manages access to data sources, and manages data integrity.

5.2.6 NDM (Network Exposure Function)

NDM is responsible for network management, including network and access management. It manages the UDM and connects to its data source.

5.2.7 UDM (User Plane Service Function)

The UDM is responsible for network management, including the UDM's access control, network security, and network access management. It manages the UDM and connects to its data source.

5.2.8 AUSF (Netwo

### Hugging Face Authentication

Llama-2 models are gated on Hugging Face. You'll need to provide your Hugging Face authentication token to download and use them.

1.  Go to [Hugging Face Settings/Tokens](https://huggingface.co/settings/tokens).
2.  Create a new token with at least `read` access.
3.  In Colab, add this token to the secrets manager under the "🔑" icon in the left panel. Name the secret `HF_TOKEN`.

In [3]:
# Authenticate with Hugging Face
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=True)
print("Logged in to Hugging Face.")

Logged in to Hugging Face.


In [4]:
# Install required libraries for Llama-2 and 4-bit quantization
!pip install -q bitsandbytes accelerate transformers safetensors
print("\nStep 1.1 Complete: Llama-2 dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.3 MB/s eta 0:00:00

Step 1.1 Complete: Llama-2 dependencies installed.


In [5]:
!pip install -U bitsandbytes>=0.46.1
print("bitsandbytes updated to >=0.46.1")

bitsandbytes updated to >=0.46.1


In [6]:
import json
import re # Import regex module
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# from langchain_community.llms import HuggingFacePipeline # Old import
from langchain_huggingface.llms import HuggingFacePipeline # Updated import
import torch
from huggingface_hub import login
from google.colab import userdata

# Get HF_TOKEN from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')
print("HF_TOKEN loaded from Colab secrets.")

# Initialize the Language Model for Llama-2
model_name = "meta-llama/Llama-2-7b-chat-hf" # Using Llama-2-7b-chat-hf

# Configure 4-bit quantization
bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True # Added to resolve ValueError for offloading
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN # Use the HF_TOKEN for gated models
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bitsandbytes_config,
    device_map="auto", # Automatically maps the model to available devices (GPU)
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN # Use the HF_TOKEN for gated models
    # Removed use_flash_attention_2=False as it's not a direct argument for LlamaForCausalLM's __init__
)

# Create the text-generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # device_map="auto" handled by model loading, no need for 'device'
    return_full_text=False, # Crucial: only return generated text, not the prompt
    max_new_tokens=1000, # Increased for Llama-2's capability
    temperature=0.7, # Add a temperature for more diverse outputs
    top_p=0.9 # Top-p sampling
)

llm = HuggingFacePipeline(pipeline=generator)
print(f"\nLanguage model '{model_name}' initialized.")

# Define the prompt template
qna_prompt_template = """[INST] You are an expert in 3GPP telecommunications.
Your task is to read the following 3GPP specification text and generate exactly 10 question-answer pairs based on the content.
The questions should be specific and the answers should be concise and directly derived from the provided text.
Make sure the questions cover different aspects of the text.

Text: {text}

Output ONLY the JSON array. Do NOT include any other text, explanation, or conversational elements outside the JSON array.
The JSON array must contain exactly 10 objects, each with 'question' and 'answer' keys.

JSON Format Example:
```json
[
  {{"question": "What is X?", "answer": "Y."}},
  {{"question": "How does A work?", "answer": "B."}}
]
```
[/INST]
"""

# Create the prompt template instance
qna_prompt = PromptTemplate(
    input_variables=["text"],
    template=qna_prompt_template
)
print("Q&A generation prompt defined.")

# Placeholder for 3GPP content. REPLACE THIS WITH YOUR ACTUAL DATA.
# You might load this from a file, scrape a webpage, or have it as a string.
example_3gpp_text = """3GPP Technical Specification 23.501 V16.8.0 (2020-12)
5G System Architecture

5.1 General Principles
The 5G System architecture is defined to support various services and use cases, including enhanced Mobile Broadband (eMBB), Ultra-Reliable Low Latency Communication (URLLC), and massive Machine Type Communication (mMTC). It builds upon the existing 4G LTE architecture with significant enhancements to meet the requirements of new services. Key architectural principles include service-based architecture, network slicing, and control/user plane separation.

5.2 Network Functions
Key Network Functions (NFs) in the 5G System include: AMF (Access and Mobility Management Function), SMF (Session Management Function), UPF (User Plane Function), UDM (Unified Data Management), AUSF (Authentication Server Function), PCF (Policy Control Function), NEF (Network Exposure Function), and NRF (NF Repository Function).

5.2.1 AMF (Access and Mobility Management Function)
The AMF is responsible for Connection Management (CM) and Mobility Management (MM). It handles UE registration, connection setup, reachability management, and mobility between 3GPP access and non-3GPP access. The AMF also performs authentication and authorization with the AUSF and UDM.

5.2.2 SMF (Session Management Function)
The SMF is responsible for session management, including session establishment, modification, and release. It selects and controls the UPF, allocates IP addresses, and manages QoS flows.

5.2.3 UPF (User Plane Function)
The UPF is the core component for the user plane data handling. It performs packet routing and forwarding, inter-system mobility, and acts as an anchor point for session continuity.
"""

# Use the LLM to generate Q&A pairs
print("Generating Q&A pairs...")

# Format the prompt with the text
formatted_prompt = qna_prompt.format(text=example_3gpp_text)

# Generate the response using the invoke method
raw_response = llm.invoke(formatted_prompt)

# --- Start of Modified Parsing Logic ---
qna_pairs = []

# Try to extract the JSON block using markdown code block syntax if the model adheres to it
json_block_match = re.search(r'```json\s*(\[.*?\])\s*```', raw_response, re.DOTALL)

if json_block_match:
    json_string = json_block_match.group(1)
    try:
        qna_pairs = json.loads(json_string)
        # Validate that each item in the list is a dict with 'question' and 'answer'
        if not all(isinstance(item, dict) and 'question' in item and 'answer' in item for item in qna_pairs):
            print("Warning: JSON block was found but its structure is not as expected. Attempting regex fallback.")
            qna_pairs = [] # Reset to try regex fallback
    except json.JSONDecodeError as e:
        print(f"Warning: Failed to parse extracted JSON block with json.loads: {e}. Attempting regex fallback.")
        qna_pairs = [] # Reset to try regex fallback
else:
    print("No `json` markdown block found. Attempting direct JSON parsing or regex fallback.")

if not qna_pairs:
    # Fallback to the original regex approach if direct json.loads or markdown block extraction fails
    # Regex to find all occurrences of {"question": "...", "answer": "..."}
    # It's made more robust by handling potential escaped quotes within the string values
    json_pattern = re.compile(r'{\s*"question"\s*:\s*"(.*?)(?<!\\)"\s*,\s*"answer"\s*:\s*"(.*?)(?<!\\)"\s*}', re.DOTALL)
    matches = json_pattern.findall(raw_response)

    if matches:
        for q_text, a_text in matches:
            # Unescape quotes that might have been escaped by the model within the strings
            question = q_text.replace('\"', '"')
            answer = a_text.replace('\"', '"')
            qna_pairs.append({"question": question, "answer": answer})

if qna_pairs:
    print(f"\nSuccessfully extracted {len(qna_pairs)} potential Q&A pairs:")
    for i, qa in enumerate(qna_pairs):
        print(f"Q{i+1}: {qa['question']}")
        print(f"A{i+1}: {qa['answer']}\n")
else:
    print("\nNo valid Q&A pairs could be extracted from the model output. Raw response:\n")
    print(raw_response)
# --- End of Modified Parsing Logic ---

print("\nStep 4 Complete: Q&A generation attempt finished. Review the output for quality and completeness.")

HF_TOKEN loaded from Colab secrets.


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



Language model 'meta-llama/Llama-2-7b-chat-hf' initialized.
Q&A generation prompt defined.
Generating Q&A pairs...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


No `json` markdown block found. Attempting direct JSON parsing or regex fallback.

Successfully extracted 10 potential Q&A pairs:
Q1: What is the purpose of the AMF?
A1: The AMF is responsible for Connection Management (CM) and Mobility Management (MM).

Q2: What is the role of the SMF?
A2: The SMF is responsible for session management, including session establishment, modification, and release.

Q3: What is the function of the UPF?
A3: The UPF is the core component for the user plane data handling and performs packet routing and forwarding, inter-system mobility, and acts as an anchor point for session continuity.

Q4: What are the key Network Functions in the 5G System?
A4: Key Network Functions (NFs) in the 5G System include: AMF, SMF, UPF, UDM, AUSF, PCF, NEF, and NRF.

Q5: What is the purpose of the AUSF?
A5: The AUSF performs authentication and authorization with the UDM.

Q6: What is the function of the UDM?
A6: The UDM manages the authentication and authorization of the UE.

Q7

## Data Acquisition and Curation

This section provides code examples for acquiring data from various sources (web, PDF, FTP) which is crucial for building a comprehensive dataset for your telecom LLM. These are foundational blocks; a full-fledged data pipeline would require more robust error handling, parallel processing, and domain-specific parsing rules.

In [7]:
import requests
from bs4 import BeautifulSoup

def scrape_web_page(url):
    """Scrapes text content from a given URL."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')
        # Extract text from common content areas, e.g., paragraphs, headings
        content_tags = soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])
        text_content = '\n'.join([tag.get_text() for tag in content_tags])
        return text_content
    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return None

print("Web scraping function defined. Example usage:")
# Example usage: (replace with actual 3GPP/GSMA/NTN/FCC links)
# 3GPP often provides specifications directly as PDF, so direct web scraping of text might be limited.
# However, you might find news, overview, or introductory pages useful.
example_url = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
web_text = scrape_web_page(example_url)
if web_text:
    print(f"--- Scraped content from {example_url} (first 500 chars) ---")
    print(web_text[:500] + "...")
else:
    print(f"Failed to scrape content from {example_url}.")

Web scraping function defined. Example usage:
--- Scraped content from https://www.3gpp.org/about-3gpp/what-is-3gpp (first 500 chars) ---
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project Coordination Group (PCG)
Achievement Awards


Mobile Competence CentreVacan...


In [8]:
# Install pypdf for PDF parsing (if not already installed through other deps)
!pip install -q pypdf

from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """Extracts text from a local PDF file."""
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        print(f"Error extracting text from {pdf_path}: {e}")
        return None

print("PDF text extraction function defined.")
print("To use, you would first need to download a PDF file to the Colab environment.")
print("Example: !wget -q 'https://www.3gpp.org/ftp/Specs/archive/23_series/23.501/23501-g80.zip' -O 23501.zip")
print("Then unzip: !unzip -o 23501.zip")
print("Then you can call: extract_text_from_pdf('23501-g80.pdf')")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 10.5 MB/s eta 0:00:00
PDF text extraction function defined.
To use, you would first need to download a PDF file to the Colab environment.
Example: !wget -q 'https://www.3gpp.org/ftp/Specs/archive/23_series/23.501/23501-g80.zip' -O 23501.zip
Then unzip: !unzip -o 23501.zip
Then you can call: extract_text_from_pdf('23501-g80.pdf')


In [9]:
from ftplib import FTP

def download_from_ftp(ftp_server, username, password, remote_path, local_path):
    """Downloads a file from an FTP server."""
    try:
        with FTP(ftp_server) as ftp:
            ftp.login(user=username, passwd=password)
            with open(local_path, 'wb') as local_file:
                ftp.retrbinary(f'RETR {remote_path}', local_file.write)
        print(f"Successfully downloaded {remote_path} to {local_path}")
        return True
    except Exception as e:
        print(f"Error downloading from FTP {ftp_server}/{remote_path}: {e}")
        return False

print("FTP download function defined.")
print("Note: 3GPP specifications are primarily available via HTTP/HTTPS, but FTP might be relevant for other data sources.")
print("You would need server details, username, password, and the specific file paths to use this function.")
print("Example: download_from_ftp('ftp.example.com', 'user', 'pass', '/path/to/remote/file.txt', 'local_file.txt')")

FTP download function defined.
Note: 3GPP specifications are primarily available via HTTP/HTTPS, but FTP might be relevant for other data sources.
You would need server details, username, password, and the specific file paths to use this function.
Example: download_from_ftp('ftp.example.com', 'user', 'pass', '/path/to/remote/file.txt', 'local_file.txt')


## Text Chunking and Processing

Once you have acquired raw text from web pages, PDFs, or FTP, it's crucial to break it down into smaller, manageable chunks. Large language models (LLMs) have a limited context window, and processing entire documents at once is often not feasible or efficient. Chunking helps ensure that relevant information fits within the model's input limit and allows for focused processing.

Here's an example using `langchain_text_splitters` to split text while trying to maintain semantic coherence.

In [10]:
!pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    """Chunks a given text into smaller, overlapping segments."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.create_documents([text])
    # The create_documents method returns a list of Document objects.
    # We extract the page_content which is the text itself.
    return [chunk.page_content for chunk in chunks]

print("Text chunking function defined.")

# Example usage with the previously scraped web_text
if 'web_text' in locals() and web_text:
    print(f"Original text length: {len(web_text)} characters.")
    text_chunks = chunk_text(web_text, chunk_size=500, chunk_overlap=100)
    print(f"Generated {len(text_chunks)} chunks.")
    if text_chunks:
        print("--- First chunk (500 chars max) ---")
        print(text_chunks[0][:500] + "...")
        if len(text_chunks) > 1:
            print("--- Second chunk (500 chars max) ---")
            print(text_chunks[1][:500] + "...")
else:
    print("No 'web_text' available from previous execution. Please run the web scraping cell first.")

print("\nNow, these chunks can be used as input for generating synthetic Q&A pairs or for fine-tuning.")

Text chunking function defined.
Original text length: 14236 characters.
Generated 41 chunks.
--- First chunk (500 chars max) ---
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project Coordination Group (PCG)
Achievement Awards...
--- Second chunk (500 chars max) ---
Mobile Competence CentreVacancies MCC Task Forces
Vacancies 
MCC Task Forces
Funding & Finances
Liaisons Statements
Legal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law
Logo Usage
3GPP privacy policy
Call for IPR
Statement Regarding Competition Law
3GPP FAQs...

Now, these chunks can be used as input for generating syntheti

## PDF Text Extraction with PyPDF2

Previously, we defined a function using `pypdf`. Here's an alternative implementation using `PyPDF2` as per your request. You might prefer one over the other based on specific PDF characteristics or library performance.

In [11]:
!pip install -q PyPDF2

from PyPDF2 import PdfReader

def extract_text_from_pdf_pypdf2(pdf_path):
    """Extracts text from a local PDF file using PyPDF2."""
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        print(f"Error extracting text from {pdf_path} using PyPDF2: {e}")
        return None

print("PDF text extraction function using PyPDF2 defined.")
print("Example usage: extract_text_from_pdf_pypdf2('your_document.pdf')")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.2 MB/s eta 0:00:00
PDF text extraction function using PyPDF2 defined.
Example usage: extract_text_from_pdf_pypdf2('your_document.pdf')


## Processing Multiple PDF Files and Chunking Text

This example demonstrates how to iterate through a list of PDF files, extract text from each using one of our defined functions (e.g., `extract_text_from_pdf`), and then apply the `chunk_text` function to each document's content. This is a common workflow for preparing large document corpuses for LLM processing.

In [12]:
# Since direct PDF downloads from 3GPP FTP are facing '403 Forbidden' errors,
# we will use the previously defined `scrape_web_page` function to get some text
# and demonstrate the chunking process.
# The `scrape_web_page` function is defined in cell 'ef238acf' and should be available.
# The `chunk_text` function is defined in cell 'f7fe4408' and should be available.
# The `extract_text_from_pdf` function is defined in cell '6e42eb1c' and should be available.

# Using the example_url from a previous cell ('ef238acf') or defining a new one if needed
example_url_for_chunking = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
print(f"Attempting to scrape content from: {example_url_for_chunking}")

# Ensure `scrape_web_page` is defined and accessible
# It was defined in cell 'ef238acf'
if 'scrape_web_page' not in globals():
    print("Error: `scrape_web_page` function not found. Please ensure cell 'ef238acf' has been run.")
    # Fallback or exit if function is truly not available
    # For this demonstration, we'll assume it exists if the previous cell executed.

scraped_text_for_chunking = scrape_web_page(example_url_for_chunking)

all_document_chunks = {}

if scraped_text_for_chunking:
    print(f"Successfully scraped text (first 200 chars):\n{scraped_text_for_chunking[:200]}...")
    document_name = "3gpp_about_page"
    print(f"\nProcessing {document_name}...")

    # Chunk the extracted text
    # Ensure `chunk_text` is defined and accessible
    # It was defined in cell 'f7fe4408'
    if 'chunk_text' not in globals():
        print("Error: `chunk_text` function not found. Please ensure cell 'f7fe4408' has been run.")
    else:
        chunks = chunk_text(scraped_text_for_chunking, chunk_size=1000, chunk_overlap=200)
        all_document_chunks[document_name] = chunks
        print(f"Generated {len(chunks)} chunks for {document_name}.")
        if chunks:
            print(f"First chunk of {document_name}:\n{chunks[0][:500]}...") # Display first 500 chars of first chunk
        else:
            print(f"No chunks generated for {document_name}.")
else:
    print(f"Could not scrape text from {example_url_for_chunking}. Skipping chunking.")

print("\nFinished processing all specified content.")
print(f"Total documents (or web pages) processed for chunking: {len(all_document_chunks)}")

# You can now access chunks, e.g., all_document_chunks['3gpp_about_page']

Attempting to scrape content from: https://www.3gpp.org/about-3gpp/what-is-3gpp
Successfully scraped text (first 200 chars):
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence Cent...

Processing 3gpp_about_page...
Generated 20 chunks for 3gpp_about_page.
First chunk of 3gpp_about_page:
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project Coordination Group (PCG)
Achievement Awards...

Finished processing all specified content.
Total documents (or web pages) processed for chunki

## Generate Synthetic Q&A Pairs from Chunks

Now that we have successfully chunked the text from the scraped webpage, we can iterate through each chunk and use our fine-tuned Llama-2 model to generate synthetic Question & Answer pairs. This process will help in building a comprehensive dataset for instruction tuning.

In [13]:
# Install required library for text splitting (moved from d713a92c)
!pip install -q langchain-text-splitters
print("Installed `langchain-text-splitters`.")

Installed `langchain-text-splitters`.


### Re-initializing `all_document_chunks` and Generating Q&A Pairs (Revised)

To address the persistent `NameError` for `all_document_chunks`, we'll explicitly recreate it and then generate the Q&A pairs in a guaranteed sequence. This will ensure `all_document_chunks` is in scope for the Q&A generation process.

In [14]:
# Re-execute logic for creating `all_document_chunks` (from d713a92c)

# The `scrape_web_page` and `chunk_text` functions are assumed to be defined by prior cells (ef238acf, f7fe4408).
# If they are not, please run those cells first.

example_url_for_chunking = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
print(f"Attempting to scrape content from: {example_url_for_chunking}")

scraped_text_for_chunking = scrape_web_page(example_url_for_chunking)

all_document_chunks = {}

if scraped_text_for_chunking:
    print(f"Successfully scraped text (first 200 chars):\n{scraped_text_for_chunking[:200]}...")
    document_name = "3gpp_about_page"
    print(f"\nProcessing {document_name}...")

    chunks = chunk_text(scraped_text_for_chunking, chunk_size=1000, chunk_overlap=200)
    all_document_chunks[document_name] = chunks
    print(f"Generated {len(chunks)} chunks for {document_name}.")
    if chunks:
        print(f"First chunk of {document_name}:\n{chunks[0][:500]}...")
    else:
        print(f"No chunks generated for {document_name}.")
else:
    print(f"Could not scrape text from {example_url_for_chunking}. Skipping chunking.")

print("\nFinished processing all specified content for chunking.")
print(f"Total documents (or web pages) processed for chunking: {len(all_document_chunks)}")

Attempting to scrape content from: https://www.3gpp.org/about-3gpp/what-is-3gpp
Successfully scraped text (first 200 chars):
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence Cent...

Processing 3gpp_about_page...
Generated 20 chunks for 3gpp_about_page.
First chunk of 3gpp_about_page:
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project Coordination Group (PCG)
Achievement Awards...

Finished processing all specified content for chunking.
Total documents (or web pages) process

In [15]:
# Re-execute logic for generating synthetic Q&A pairs (from a4225127)
import json
import re

# List to store all generated Q&A pairs from all chunks
all_generated_qna_pairs = []

print("Starting Q&A generation for all chunks...")

# Ensure all_document_chunks is not empty before proceeding
if not all_document_chunks:
    print("Error: `all_document_chunks` is empty or not defined. Q&A generation cannot proceed.")
else:
    # Iterate through each document's chunks
    for doc_name, chunks in all_document_chunks.items():
        print(f"\n--- Processing chunks for document: {doc_name} ---")
        for i, chunk in enumerate(chunks):
            print(f"  Processing chunk {i+1}/{len(chunks)} (length: {len(chunk)} chars)...")

            # Ensure qna_prompt and llm are defined from previous cells (d2bec2a0)
            if 'qna_prompt' not in globals() or 'llm' not in globals():
                print("    Error: `qna_prompt` or `llm` not found. Please ensure the LLM initialization cell (d2bec2a0) has been run.")
                break # Exit inner loop

            # Format the prompt with the current chunk's text
            formatted_prompt = qna_prompt.format(text=chunk)

            try:
                # Generate the response using the invoke method
                raw_response = llm.invoke(formatted_prompt)

                chunk_qna_pairs = []

                # Try to extract the JSON block using markdown code block syntax
                json_block_match = re.search(r'```json\s*(\[.*?\])\s*```', raw_response, re.DOTALL)

                if json_block_match:
                    json_string = json_block_match.group(1)
                    try:
                        parsed_json = json.loads(json_string)
                        if all(isinstance(item, dict) and 'question' in item and 'answer' in item for item in parsed_json):
                            chunk_qna_pairs = parsed_json
                        else:
                            print(f"    Warning: JSON block for chunk {i+1} found but its structure is not as expected. Attempting regex fallback.")
                    except json.JSONDecodeError as e:
                        print(f"    Warning: Failed to parse extracted JSON block for chunk {i+1} with json.loads: {e}. Attempting regex fallback.")

                if not chunk_qna_pairs:
                    json_pattern = re.compile(r'\{\s*"question"\s*:\s*"(.*?)(?<!\\)"\s*,\s*"answer"\s*:\s*"(.*?)(?<!\\)"\s*\}', re.DOTALL)
                    matches = json_pattern.findall(raw_response)

                    if matches:
                        for q_text, a_text in matches:
                            question = q_text.replace('\"', '"')
                            answer = a_text.replace('\"', '"')
                            chunk_qna_pairs.append({"question": question, "answer": answer})

                if chunk_qna_pairs:
                    all_generated_qna_pairs.extend(chunk_qna_pairs)
                    print(f"    Successfully extracted {len(chunk_qna_pairs)} Q&A pairs from chunk {i+1}.")
                else:
                    print(f"    No valid Q&A pairs could be extracted from chunk {i+1}. Raw response (first 500 chars):\n{raw_response[:500]}...")

            except Exception as e:
                print(f"    Error generating Q&A for chunk {i+1}: {e}")

print("\nFinished Q&A generation for all chunks.")
print(f"Total Q&A pairs generated across all documents: {len(all_generated_qna_pairs)}")

if all_generated_qna_pairs:
    print("\n--- First 5 generated Q&A pairs ---")
    for i, qa in enumerate(all_generated_qna_pairs[:5]):
        print(f"Q{i+1}: {qa['question']}")
        print(f"A{i+1}: {qa['answer']}\n")
    if len(all_generated_qna_pairs) > 5:
        print(f"... and {len(all_generated_qna_pairs) - 5} more Q&A pairs.")
else:
    print("No Q&A pairs were successfully generated.")

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Q&A generation for all chunks...

--- Processing chunks for document: 3gpp_about_page ---
  Processing chunk 1/20 (length: 468 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 9 Q&A pairs from chunk 1.
  Processing chunk 2/20 (length: 930 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    Successfully extracted 9 Q&A pairs from chunk 17.
  Processing chunk 18/20 (length: 996 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 10 Q&A pairs from chunk 18.
  Processing chunk 19/20 (length: 885 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 9 Q&A pairs from chunk 19.
  Processing chunk 20/20 (length: 225 chars)...
    Successfully extracted 1 Q&A pairs from chunk 20.

Finished Q&A generation for all chunks.
Total Q&A pairs generated across all documents: 183

--- First 5 generated Q&A pairs ---
Q1: What is the 3GPP Portal?
A1: The 3GPP Portal is a website that provides information and resources for 3GPP members and stakeholders.

Q2: What is the Project Coordination Group (PCG)?
A2: The PCG is a group within 3GPP that coordinates the development of mobile standards.

Q3: What are the Achievement Awards?
A3: The Achievement Awards are given to individuals or organizations that have made significant contributions to 3GPP.

Q4: What is the Mobile Competence Centre?
A4: The Mobile Competence Centre is a facility within 3GPP that provides testing and validation services for mobile devices and networks.

Q5: What are the Membership categories in 3GPP?
A5: There are several membership categories in 3GP

In [16]:
# Save all_generated_qna_pairs to a JSON file
import json
import os

output_filename = "generated_qna_pairs.json"

if all_generated_qna_pairs:
    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(all_generated_qna_pairs, f, ensure_ascii=False, indent=4)
        print(f"Successfully saved {len(all_generated_qna_pairs)} Q&A pairs to '{output_filename}'.")
        print(f"File path: {os.path.abspath(output_filename)}")
    except Exception as e:
        print(f"Error saving Q&A pairs to JSON file: {e}")
else:
    print("No Q&A pairs were generated to save.")

Successfully saved 183 Q&A pairs to 'generated_qna_pairs.json'.
File path: /content/generated_qna_pairs.json


### Re-initializing `all_document_chunks` and Generating Q&A Pairs

It seems the `all_document_chunks` variable was not available in the current kernel session when the Q&A generation was attempted. This might happen after a kernel restart or if the cell defining it was not executed.

To ensure `all_document_chunks` is properly defined and to continue with the Q&A generation, I will re-execute the code for processing and chunking the web content, followed by the Q&A generation code.

In [17]:
# --- BEGIN REQUIRED FUNCTION DEFINITIONS (pulled from ef238acf and f7fe4408) ---
import requests
from bs4 import BeautifulSoup

def scrape_web_page(url):
    """Scrapes text content from a given URL."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')
        content_tags = soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])
        text_content = '\n'.join([tag.get_text() for tag in content_tags])
        return text_content
    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return None

# Install if not already installed (idempotent)
!pip install -q langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    """Chunks a given text into smaller, overlapping segments."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.create_documents([text])
    return [chunk.page_content for chunk in chunks]
# --- END REQUIRED FUNCTION DEFINITIONS ---


# Code from original cell 'd36218e5' to re-initialize `all_document_chunks`

# Since direct PDF downloads from 3GPP FTP are facing '403 Forbidden' errors,
# we will use the previously defined `scrape_web_page` function to get some text
# and demonstrate the chunking process.

# Using the example_url from a previous cell ('ef238acf') or defining a new one if needed
example_url_for_chunking = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
print(f"Attempting to scrape content from: {example_url_for_chunking}")

scraped_text_for_chunking = scrape_web_page(example_url_for_chunking)

all_document_chunks = {}

if scraped_text_for_chunking:
    print(f"Successfully scraped text (first 200 chars):\n{scraped_text_for_chunking[:200]}...")
    document_name = "3gpp_about_page"
    print(f"\nProcessing {document_name}...")

    # Chunk the extracted text
    chunks = chunk_text(scraped_text_for_chunking, chunk_size=1000, chunk_overlap=200)
    all_document_chunks[document_name] = chunks
    print(f"Generated {len(chunks)} chunks for {document_name}.")
    if chunks:
        print(f"First chunk of {document_name}:\n{chunks[0][:500]}...") # Display first 500 chars of first chunk
    else:
        print(f"No chunks generated for {document_name}.")
else:
    print(f"Could not scrape text from {example_url_for_chunking}. Skipping chunking.")

print("\nFinished processing all specified content.")
print(f"Total documents (or web pages) processed for chunking: {len(all_document_chunks)}")

# You can now access chunks, e.g., all_document_chunks['3gpp_about_page']

Attempting to scrape content from: https://www.3gpp.org/about-3gpp/what-is-3gpp
Successfully scraped text (first 200 chars):
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence Cent...

Processing 3gpp_about_page...
Generated 20 chunks for 3gpp_about_page.
First chunk of 3gpp_about_page:
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project Coordination Group (PCG)
Achievement Awards...

Finished processing all specified content.
Total documents (or web pages) processed for chunki

In [18]:
# Code from original cell '61a61728' to generate synthetic Q&A pairs
import json
import re

# List to store all generated Q&A pairs from all chunks
all_generated_qna_pairs = []

print("Starting Q&A generation for all chunks...")

# Iterate through each document's chunks (though currently only one document here)
for doc_name, chunks in all_document_chunks.items():
    print(f"\n--- Processing chunks for document: {doc_name} ---")
    for i, chunk in enumerate(chunks):
        print(f"  Processing chunk {i+1}/{len(chunks)} (length: {len(chunk)} chars)...")

        # Format the prompt with the current chunk's text
        formatted_prompt = qna_prompt.format(text=chunk)

        try:
            # Generate the response using the invoke method
            raw_response = llm.invoke(formatted_prompt)

            # --- Parsing Logic (copied from previous Q&A generation cell) ---
            chunk_qna_pairs = []

            # Try to extract the JSON block using markdown code block syntax
            json_block_match = re.search(r'```json\s*(\[.*?\])\s*```', raw_response, re.DOTALL)

            if json_block_match:
                json_string = json_block_match.group(1)
                try:
                    parsed_json = json.loads(json_string)
                    # Validate that each item in the list is a dict with 'question' and 'answer'
                    if all(isinstance(item, dict) and 'question' in item and 'answer' in item for item in parsed_json):
                        chunk_qna_pairs = parsed_json
                    else:
                        print(f"    Warning: JSON block for chunk {i+1} found but its structure is not as expected. Attempting regex fallback.")
                except json.JSONDecodeError as e:
                    print(f"    Warning: Failed to parse extracted JSON block for chunk {i+1} with json.loads: {e}. Attempting regex fallback.")
            else:
                # print(f"    No `json` markdown block found for chunk {i+1}. Attempting direct JSON parsing or regex fallback.")
                pass # Suppress this print for cleaner output during iteration

            if not chunk_qna_pairs:
                # Fallback to the original regex approach if direct json.loads or markdown block extraction fails
                json_pattern = re.compile(r'\{\s*"question"\s*:\s*"(.*?)(?<!\\)"\s*,\s*"answer"\s*:\s*"(.*?)(?<!\\)"\s*\}', re.DOTALL)
                matches = json_pattern.findall(raw_response)

                if matches:
                    for q_text, a_text in matches:
                        question = q_text.replace('\"', '"')
                        answer = a_text.replace('\"', '"')
                        chunk_qna_pairs.append({"question": question, "answer": answer})

            if chunk_qna_pairs:
                all_generated_qna_pairs.extend(chunk_qna_pairs)
                print(f"    Successfully extracted {len(chunk_qna_pairs)} Q&A pairs from chunk {i+1}.")
            else:
                print(f"    No valid Q&A pairs could be extracted from chunk {i+1}. Raw response (first 500 chars):\n{raw_response[:500]}...")
            # --- End of Parsing Logic ---

        except Exception as e:
            print(f"    Error generating Q&A for chunk {i+1}: {e}")

print("\nFinished Q&A generation for all chunks.")
print(f"Total Q&A pairs generated across all documents: {len(all_generated_qna_pairs)}")

if all_generated_qna_pairs:
    print("\n--- First 5 generated Q&A pairs ---")
    for i, qa in enumerate(all_generated_qna_pairs[:5]):
        print(f"Q{i+1}: {qa['question']}")
        print(f"A{i+1}: {qa['answer']}\n")
    if len(all_generated_qna_pairs) > 5:
        print(f"... and {len(all_generated_qna_pairs) - 5} more Q&A pairs.")
else:
    print("No Q&A pairs were successfully generated.")

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Q&A generation for all chunks...

--- Processing chunks for document: 3gpp_about_page ---
  Processing chunk 1/20 (length: 468 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    No valid Q&A pairs could be extracted from chunk 1. Raw response (first 500 chars):
Sure, here are 10 question-answer pairs based on the provided 3GPP specification text:

1. Question: What is the purpose of the 3GPP Portal?
Answer: The 3GPP Portal provides information about 3GPP, its Home, and its various activities.
2. Question: Who is eligible to become a member of 3GPP?
Answer: Any organization or individual interested in contributing to the development of 3GPP standards can become a member.
3. Question: What is the Project Coordination Group (PCG)?
Answer: The PCG is respo...
  Processing chunk 2/20 (length: 930 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    Successfully extracted 10 Q&A pairs from chunk 17.
  Processing chunk 18/20 (length: 996 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 10 Q&A pairs from chunk 18.
  Processing chunk 19/20 (length: 885 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 10 Q&A pairs from chunk 19.
  Processing chunk 20/20 (length: 220 chars)...
    No valid Q&A pairs could be extracted from chunk 20. Raw response (first 500 chars):
Here are 10 question-answer pairs based on the provided 3GPP specification text:

Question 1: What is the purpose of the "Enter your name" field?
Answer: The "Enter your name" field is used to collect the name of the user.

Question 2: What is the purpose of the "Enter your email address" field?
Answer: The "Enter your email address" field is used to collect the email address of the user.

Question 3: What does checking the box next to "Subscribe" mean?
Answer: Checking the box next to "Subscrib...

Finished Q&A generation for all chunks.
Total Q&A pairs generated across all documents: 174

--- First 5 generated Q&A pairs ---
Q1: What is the purpose of the Mobile Competence Centre?
A1: The Mobile Competence Centre is a central hub for 3GPP related activities.

Q2: What are the different types of M

**Important:** Ensure that the cell above this (where `all_document_chunks` is populated by scraping and chunking) has been executed successfully before running the Q&A generation code.

In [19]:
import json
import re

# List to store all generated Q&A pairs from all chunks
all_generated_qna_pairs = []

print("Starting Q&A generation for all chunks...")

# Iterate through each document's chunks (though currently only one document here)
for doc_name, chunks in all_document_chunks.items():
    print(f"\n--- Processing chunks for document: {doc_name} ---")
    for i, chunk in enumerate(chunks):
        print(f"  Processing chunk {i+1}/{len(chunks)} (length: {len(chunk)} chars)...")

        # Format the prompt with the current chunk's text
        formatted_prompt = qna_prompt.format(text=chunk)

        try:
            # Generate the response using the invoke method
            raw_response = llm.invoke(formatted_prompt)

            # --- Parsing Logic (copied from previous Q&A generation cell) ---
            chunk_qna_pairs = []

            # Try to extract the JSON block using markdown code block syntax
            json_block_match = re.search(r'```json\s*(\[.*?\])\s*```', raw_response, re.DOTALL)

            if json_block_match:
                json_string = json_block_match.group(1)
                try:
                    parsed_json = json.loads(json_string)
                    # Validate that each item in the list is a dict with 'question' and 'answer'
                    if all(isinstance(item, dict) and 'question' in item and 'answer' in item for item in parsed_json):
                        chunk_qna_pairs = parsed_json
                    else:
                        print(f"    Warning: JSON block for chunk {i+1} found but its structure is not as expected. Attempting regex fallback.")
                except json.JSONDecodeError as e:
                    print(f"    Warning: Failed to parse extracted JSON block for chunk {i+1} with json.loads: {e}. Attempting regex fallback.")
            else:
                # print(f"    No `json` markdown block found for chunk {i+1}. Attempting direct JSON parsing or regex fallback.")
                pass # Suppress this print for cleaner output during iteration

            if not chunk_qna_pairs:
                # Fallback to the original regex approach if direct json.loads or markdown block extraction fails
                json_pattern = re.compile(r'\{\s*"question"\s*:\s*"(.*?)(?<!\\)"\s*,\s*"answer"\s*:\s*"(.*?)(?<!\\)"\s*\}', re.DOTALL)
                matches = json_pattern.findall(raw_response)

                if matches:
                    for q_text, a_text in matches:
                        question = q_text.replace('\"', '"')
                        answer = a_text.replace('\"', '"')
                        chunk_qna_pairs.append({"question": question, "answer": answer})

            if chunk_qna_pairs:
                all_generated_qna_pairs.extend(chunk_qna_pairs)
                print(f"    Successfully extracted {len(chunk_qna_pairs)} Q&A pairs from chunk {i+1}.")
            else:
                print(f"    No valid Q&A pairs could be extracted from chunk {i+1}. Raw response (first 500 chars):\n{raw_response[:500]}...")
            # --- End of Parsing Logic ---

        except Exception as e:
            print(f"    Error generating Q&A for chunk {i+1}: {e}")

print("\nFinished Q&A generation for all chunks.")
print(f"Total Q&A pairs generated across all documents: {len(all_generated_qna_pairs)}")

if all_generated_qna_pairs:
    print("\n--- First 5 generated Q&A pairs ---")
    for i, qa in enumerate(all_generated_qna_pairs[:5]):
        print(f"Q{i+1}: {qa['question']}")
        print(f"A{i+1}: {qa['answer']}\n")
    if len(all_generated_qna_pairs) > 5:
        print(f"... and {len(all_generated_qna_pairs) - 5} more Q&A pairs.")
else:
    print("No Q&A pairs were successfully generated.")


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Q&A generation for all chunks...

--- Processing chunks for document: 3gpp_about_page ---
  Processing chunk 1/20 (length: 468 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 10 Q&A pairs from chunk 1.
  Processing chunk 2/20 (length: 930 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    Successfully extracted 10 Q&A pairs from chunk 17.
  Processing chunk 18/20 (length: 996 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 9 Q&A pairs from chunk 18.
  Processing chunk 19/20 (length: 885 chars)...


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Successfully extracted 10 Q&A pairs from chunk 19.
  Processing chunk 20/20 (length: 220 chars)...
    No valid Q&A pairs could be extracted from chunk 20. Raw response (first 500 chars):
Of course! Here are 10 question-answer pairs based on the provided 3GPP specification text:

Question 1: What is the purpose of the "Enter your name" field?
Answer: The "Enter your name" field is used to collect the name of the user.

Question 2: What is the purpose of the "Enter your email address" field?
Answer: The "Enter your email address" field is used to collect the email address of the user.

Question 3: What does checking the box next to "Subscribe" imply?
Answer: Checking the box next ...

Finished Q&A generation for all chunks.
Total Q&A pairs generated across all documents: 178

--- First 5 generated Q&A pairs ---
Q1: What is the 3GPP Portal?
A1: The 3GPP Portal is a website providing information on 3GPP technologies and standards.

Q2: What is the purpose of the Project Coordination G

## Next Steps

These `all_generated_qna_pairs` now form the basis of your synthetic dataset. You can save this dataset (e.g., to a JSON file) and then proceed to:

1.  **Review and Refine:** Manually inspect a sample of these pairs to ensure quality, relevance, and accuracy.
2.  **Instruction Tuning:** Use this dataset to fine-tune your Llama-2 model (or a smaller, more specialized model) to improve its performance on telecom-specific Q&A tasks.
3.  **Data Augmentation:** Generate more synthetic data from a broader range of 3GPP, GSMA, NTN, and FCC documents to expand your dataset.
4.  **Retrieval-Augmented Generation (RAG):** Integrate a retrieval component to find relevant chunks before generating answers, which is crucial for grounding the LLM's responses in specific documents.

# Task
The main goal of this project is to develop an instruction-tuned Large Language Model (LLM) capable of generating accurate Question & Answer (QnA) pairs from 3GPP (3rd Generation Partnership Project) technical specifications and related telecommunications content. This involves several key steps:

1.  **Environment Setup**: Install necessary libraries and authenticate with Hugging Face for model access.
2.  **LLM Initialization**: Initialize a suitable Large Language Model (e.g., Llama-2-7b-chat-hf) for Q&A generation.
3.  **Data Acquisition and Preprocessing**: Implement methods to scrape web content, extract text from PDFs, and chunk the text into manageable segments for LLM processing.
4.  **Synthetic Q&A Generation**: Use the initialized LLM to generate synthetic Q&A pairs from the processed text chunks.
5.  **Q&A Pair Storage**: Save the generated Q&A pairs for future use.
6.  **Next Steps Outline**: Plan the subsequent phases, including reviewing and refining the Q&A data, instruction tuning the LLM, data augmentation, and integrating Retrieval-Augmented Generation (RAG).

The overall objective is to create a robust pipeline for generating high-quality telecom-specific Q&A datasets that can be used to fine-tune LLMs for specialized applications in the telecommunications domain.

## Review Current Progress

### Subtask:
Summarize the successfully completed steps in the notebook, including package installations, Hugging Face authentication, LLM initialization, and data acquisition/chunking function definitions.


### Current Progress Summary

The notebook has made significant progress in setting up the environment and defining the core components for the Q&A generation pipeline:

1.  **Package Installations**: All necessary packages, including `transformers`, `langchain-community`, `bitsandbytes`, `accelerate`, `safetensors`, `pypdf`, `PyPDF2`, and `langchain-text-splitters`, have been successfully installed. A `requests` dependency conflict warning was noted during the initial installation but did not block the core functionality.

2.  **Hugging Face Authentication**: Authentication with Hugging Face was successfully completed using a token retrieved from Colab secrets, enabling access to gated models like Llama-2.

3.  **LLM Initialization**: The `meta-llama/Llama-2-7b-chat-hf` model has been successfully initialized using 4-bit quantization (`BitsAndBytesConfig`) and integrated with `HuggingFacePipeline`. The Q&A generation prompt (`qna_prompt_template`) has been defined with Llama-2 specific instructions (using `[INST]...[/INST]`).

4.  **Data Acquisition and Preprocessing Functions**:
    *   A `scrape_web_page` function was defined and demonstrated to extract text from a given URL (e.g., `https://www.3gpp.org/about-3gpp/what-is-3gpp`).
    *   `extract_text_from_pdf` (using `pypdf`) and `extract_text_from_pdf_pypdf2` (using `PyPDF2`) functions were defined for PDF text extraction, though not fully demonstrated due to local file dependency.
    *   A `download_from_ftp` function was defined for FTP downloads.
    *   A `chunk_text` function using `RecursiveCharacterTextSplitter` was defined and successfully demonstrated by chunking the scraped web content.

5.  **Status of Q&A Generation**: The attempt to generate Q&A pairs from chunks resulted in a `NameError` for `all_document_chunks` in previous executions. This indicates that while the functions are defined, the sequential flow of creating `all_document_chunks` and then using it for Q&A generation needs to be re-evaluated and properly executed to ensure all variables are in scope.

## Identify and Resolve Blockers

### Subtask:
Address the `NameError: name 'all_document_chunks' is not defined` and similar issues by re-initializing all necessary components (Hugging Face login, LLM, prompt, and chunked data) in a sequential manner to ensure proper scope and availability before Q&A generation.


**Reasoning**:
To resolve the NameError and ensure all dependencies are correctly initialized and in scope, I will create a consolidated code block that performs Hugging Face authentication, LLM setup, prompt definition, and data chunking.



In [20]:
import json
import re # Import regex module
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_community.llms import HuggingFacePipeline
import torch
from huggingface_hub import login
from google.colab import userdata
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("--- Starting full initialization process ---")

# 1. Hugging Face Authentication
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=True)
print("Logged in to Hugging Face.")

# 2. LLM Initialization for Llama-2
model_name = "meta-llama/Llama-2-7b-chat-hf"

bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bitsandbytes_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=1000,
    temperature=0.7,
    top_p=0.9
)

llm = HuggingFacePipeline(pipeline=generator)
print(f"Language model '{model_name}' initialized.")

# 3. Define the prompt template
qna_prompt_template = """[INST] You are an expert in 3GPP telecommunications.\nYour task is to read the following 3GPP specification text and generate exactly 10 question-answer pairs based on the content.\nThe questions should be specific and the answers should be concise and directly derived from the provided text.\nMake sure the questions cover different aspects of the text.\n\nText: {text}\n\nOutput ONLY the JSON array. Do NOT include any other text, explanation, or conversational elements outside the JSON array.\nThe JSON array must contain exactly 10 objects, each with 'question' and 'answer' keys.\n\nJSON Format Example:\n```json\n[\n  {{\"question\": \"What is X?\", \"answer\": \"Y.\"}},\n  {{\"question\": \"How does A work?\", \"answer\": \"B.\"}}\n]\n```\n[/INST]\n"""

qna_prompt = PromptTemplate(input_variables=["text"], template=qna_prompt_template)
print("Q&A generation prompt defined.")

# 4. Data Acquisition and Chunking Function Definitions
def scrape_web_page(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        content_tags = soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])
        text_content = '\n'.join([tag.get_text() for tag in content_tags])
        return text_content
    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return None

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.create_documents([text])
    return [chunk.page_content for chunk in chunks]
print("Web scraping and text chunking functions defined.")

# 5. Execute scraping and chunking to populate all_document_chunks
example_url_for_chunking = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
print(f"Attempting to scrape content from: {example_url_for_chunking}")

scraped_text_for_chunking = scrape_web_page(example_url_for_chunking)

all_document_chunks = {}

if scraped_text_for_chunking:
    print(f"Successfully scraped text (first 200 chars):\n{scraped_text_for_chunking[:200]}...")
    document_name = "3gpp_about_page"
    print(f"\nProcessing {document_name}...")

    chunks = chunk_text(scraped_text_for_chunking, chunk_size=1000, chunk_overlap=200)
    all_document_chunks[document_name] = chunks
    print(f"Generated {len(chunks)} chunks for {document_name}.")
    if chunks:
        print(f"First chunk of {document_name}:\n{chunks[0][:500]}...")
    else:
        print(f"No chunks generated for {document_name}.")
else:
    print(f"Could not scrape text from {example_url_for_chunking}. Skipping chunking.")

print(f"Total documents (or web pages) processed for chunking: {len(all_document_chunks)}")
print("--- Full initialization process complete. All necessary variables should now be in scope. ---")


--- Starting full initialization process ---
Logged in to Hugging Face.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Language model 'meta-llama/Llama-2-7b-chat-hf' initialized.
Q&A generation prompt defined.
Web scraping and text chunking functions defined.
Attempting to scrape content from: https://www.3gpp.org/about-3gpp/what-is-3gpp
Successfully scraped text (first 200 chars):
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence Cent...

Processing 3gpp_about_page...
Generated 20 chunks for 3gpp_about_page.
First chunk of 3gpp_about_page:
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project C

**Reasoning**:
I need to ensure `langchain-community` is installed before importing it in the consolidated initialization block. Adding `!pip install -q langchain-community` at the start of the cell will address the `ModuleNotFoundError`.



In [21]:
import json
import re # Import regex module
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from huggingface_hub import login
from google.colab import userdata
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ensure langchain-huggingface is installed before import
!pip install -q langchain-huggingface
# The previous installation of langchain-community is likely not needed for HuggingFacePipeline specifically,
# but if other parts of the notebook rely on it, it might still be beneficial to have.
# For now, focusing on the fix for HuggingFacePipeline.
from langchain_huggingface.llms import HuggingFacePipeline # Updated import

print("--- Starting full initialization process ---")

# 1. Hugging Face Authentication
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=True)
print("Logged in to Hugging Face.")

# 2. LLM Initialization for Llama-2
model_name = "meta-llama/Llama-2-7b-chat-hf"

bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bitsandbytes_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=1000,
    temperature=0.7,
    top_p=0.9
)

llm = HuggingFacePipeline(pipeline=generator)
print(f"Language model '{model_name}' initialized.")

# 3. Define the prompt template
qna_prompt_template = """[INST] You are an expert in 3GPP telecommunications.\nYour task is to read the following 3GPP specification text and generate exactly 10 question-answer pairs based on the content.\nThe questions should be specific and the answers should be concise and directly derived from the provided text.\nMake sure the questions cover different aspects of the text.\n\nText: {text}\n\nOutput ONLY the JSON array. Do NOT include any other text, explanation, or conversational elements outside the JSON array.\nThe JSON array must contain exactly 10 objects, each with 'question' and 'answer' keys.\n\nJSON Format Example:\n```json\n[\n  {{\"question\": \"What is X?\", \"answer\": \"Y.\"}},\n  {{\"question\": \"How does A work?\", \"answer\": \"B.\"}}\n]\n```\n[/INST]\n"""

qna_prompt = PromptTemplate(input_variables=["text"], template=qna_prompt_template)
print("Q&A generation prompt defined.")

# 4. Data Acquisition and Chunking Function Definitions
def scrape_web_page(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        content_tags = soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])
        text_content = '\n'.join([tag.get_text() for tag in content_tags])
        return text_content
    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return None

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.create_documents([text])
    return [chunk.page_content for chunk in chunks]
print("Web scraping and text chunking functions defined.")

# 5. Execute scraping and chunking to populate all_document_chunks
example_url_for_chunking = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
print(f"Attempting to scrape content from: {example_url_for_chunking}")

scraped_text_for_chunking = scrape_web_page(example_url_for_chunking)

all_document_chunks = {}

if scraped_text_for_chunking:
    print(f"Successfully scraped text (first 200 chars):\n{scraped_text_for_chunking[:200]}...")
    document_name = "3gpp_about_page"
    print(f"\nProcessing {document_name}...")

    chunks = chunk_text(scraped_text_for_chunking, chunk_size=1000, chunk_overlap=200)
    all_document_chunks[document_name] = chunks
    print(f"Generated {len(chunks)} chunks for {document_name}.")
    if chunks:
        print(f"First chunk of {document_name}:\n{chunks[0][:500]}...")
    else:
        print(f"No chunks generated for {document_name}.")
else:
    print(f"Could not scrape text from {example_url_for_chunking}. Skipping chunking.")

print(f"Total documents (or web pages) processed for chunking: {len(all_document_chunks)}")
print("--- Full initialization process complete. All necessary variables should now be in scope. ---")

--- Starting full initialization process ---
Logged in to Hugging Face.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Language model 'meta-llama/Llama-2-7b-chat-hf' initialized.
Q&A generation prompt defined.
Web scraping and text chunking functions defined.
Attempting to scrape content from: https://www.3gpp.org/about-3gpp/what-is-3gpp
Successfully scraped text (first 200 chars):
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence Cent...

Processing 3gpp_about_page...
Generated 20 chunks for 3gpp_about_page.
First chunk of 3gpp_about_page:
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project C

# Task
The main goal of this project is to develop an instruction-tuned Large Language Model (LLM) capable of generating accurate Question & Answer (QnA) pairs from 3GPP (3rd Generation Partnership Project) technical specifications and related telecommunications content. This involves several key steps:

1.  **Environment Setup**: Install necessary libraries and authenticate with Hugging Face for model access.
2.  **LLM Initialization**: Initialize a suitable Large Language Model (e.g., Llama-2-7b-chat-hf) for Q&A generation.
3.  **Data Acquisition and Preprocessing**: Implement methods to scrape web content, extract text from PDFs, and chunk the text into manageable segments for LLM processing.
4.  **Synthetic Q&A Generation**: Use the initialized LLM to generate synthetic Q&A pairs from the processed text chunks.
5.  **Q&A Pair Storage**: Save the generated Q&A pairs for future use.
6.  **Next Steps Outline**: Plan the subsequent phases, including reviewing and refining the Q&A data, instruction tuning the LLM, data augmentation, and integrating Retrieval-Augmented Generation (RAG).

The overall objective is to create a robust pipeline for generating high-quality telecom-specific Q&A datasets that can be used to fine-tune LLMs for specialized applications in the telecommunications domain.

## Re-initialize LLM and Populate all_document_chunks

### Subtask:
Execute the consolidated initialization cell (`b9b0a283`) to re-initialize the LLM, prompt, data acquisition and chunking functions, and populate `all_document_chunks`.


**Reasoning**:
The consolidated initialization cell `b9b0a283` contains all necessary imports, Hugging Face authentication, LLM initialization, prompt definition, and data chunking logic. Executing it will ensure all required components are correctly set up and `all_document_chunks` is populated, resolving previous `ModuleNotFoundError` and `NameError` issues.



In [22]:
import json
import re # Import regex module
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from huggingface_hub import login
from google.colab import userdata
import requests # Corrected this line
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ensure all necessary libraries are installed at the very beginning
!pip install -q bitsandbytes>=0.46.1
!pip install -q accelerate
!pip install -q safetensors
!pip install -q langchain-huggingface

# Explicitly import bitsandbytes to ensure it's loaded
import bitsandbytes

from langchain_huggingface.llms import HuggingFacePipeline # Updated import

print("--- Starting full initialization process ---")

# 1. Hugging Face Authentication
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=True)
print("Logged in to Hugging Face.")

# 2. LLM Initialization for Llama-2
model_name = "meta-llama/Llama-2-7b-chat-hf"

bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bitsandbytes_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=1000,
    temperature=0.7,
    top_p=0.9
)

llm = HuggingFacePipeline(pipeline=generator)
print(f"Language model '{model_name}' initialized.")

# 3. Define the prompt template
qna_prompt_template = """[INST] You are an expert in 3GPP telecommunications.\nYour task is to read the following 3GPP specification text and generate exactly 10 question-answer pairs based on the content.\nThe questions should be specific and the answers should be concise and directly derived from the provided text.\nMake sure the questions cover different aspects of the text.\n\nText: {text}\n\nOutput ONLY the JSON array. Do NOT include any other text, explanation, or conversational elements outside the JSON array.\nThe JSON array must contain exactly 10 objects, each with 'question' and 'answer' keys.\n\nJSON Format Example:\n```json\n[\n  {{"question": "What is X?", "answer": "Y."}},\n  {{"question": "How does A work?", "answer": "B."}}\n]\n```\n[/INST]\n"""

qna_prompt = PromptTemplate(input_variables=["text"], template=qna_prompt_template)
print("Q&A generation prompt defined.")

# 4. Data Acquisition and Chunking Function Definitions
def scrape_web_page(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        content_tags = soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])
        text_content = '\n'.join([tag.get_text() for tag in content_tags])
        return text_content
    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return None

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.create_documents([text])
    return [chunk.page_content for chunk in chunks]
print("Web scraping and text chunking functions defined.")

# 5. Execute scraping and chunking to populate all_document_chunks
example_url_for_chunking = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
print(f"Attempting to scrape content from: {example_url_for_chunking}")

scraped_text_for_chunking = scrape_web_page(example_url_for_chunking)

all_document_chunks = {}

if scraped_text_for_chunking:
    print(f"Successfully scraped text (first 200 chars):\n{scraped_text_for_chunking[:200]}...")
    document_name = "3gpp_about_page"
    print(f"\nProcessing {document_name}...")

    chunks = chunk_text(scraped_text_for_chunking, chunk_size=1000, chunk_overlap=200)
    all_document_chunks[document_name] = chunks
    print(f"Generated {len(chunks)} chunks for {document_name}.")
    if chunks:
        print(f"First chunk of {document_name}:\n{chunks[0][:500]}...")
    else:
        print(f"No chunks generated for {document_name}.")
else:
    print(f"Could not scrape text from {example_url_for_chunking}. Skipping chunking.")

print(f"Total documents (or web pages) processed for chunking: {len(all_document_chunks)}")
print("--- Full initialization process complete. All necessary variables should now be in scope. ---")

--- Starting full initialization process ---
Logged in to Hugging Face.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Language model 'meta-llama/Llama-2-7b-chat-hf' initialized.
Q&A generation prompt defined.
Web scraping and text chunking functions defined.
Attempting to scrape content from: https://www.3gpp.org/about-3gpp/what-is-3gpp
Successfully scraped text (first 200 chars):
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence Cent...

Processing 3gpp_about_page...
Generated 20 chunks for 3gpp_about_page.
First chunk of 3gpp_about_page:
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project C

**Reasoning**:
Adding `bitsandbytes` and other essential libraries installation at the top of the consolidated initialization block ensures they are present and correctly configured before any imports or model loading operations, which should resolve the `ImportError`.



In [23]:
import json
import re # Import regex module
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from huggingface_hub import login
from google.colab import userdata
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ensure all necessary libraries are installed at the very beginning
!pip install -q bitsandbytes>=0.46.1
!pip install -q accelerate
!pip install -q safetensors
!pip install -q langchain-huggingface

# The previous installation of langchain-community is likely not needed for HuggingFacePipeline specifically,
# but if other parts of the notebook rely on it, it might still be beneficial to have.
# For now, focusing on the fix for HuggingFacePipeline.
from langchain_huggingface.llms import HuggingFacePipeline # Updated import

print("--- Starting full initialization process ---")

# 1. Hugging Face Authentication
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=True)
print("Logged in to Hugging Face.")

# 2. LLM Initialization for Llama-2
model_name = "meta-llama/Llama-2-7b-chat-hf"

bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bitsandbytes_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=1000,
    temperature=0.7,
    top_p=0.9
)

llm = HuggingFacePipeline(pipeline=generator)
print(f"Language model '{model_name}' initialized.")

# 3. Define the prompt template
qna_prompt_template = """[INST] You are an expert in 3GPP telecommunications.\nYour task is to read the following 3GPP specification text and generate exactly 10 question-answer pairs based on the content.\nThe questions should be specific and the answers should be concise and directly derived from the provided text.\nMake sure the questions cover different aspects of the text.\n\nText: {text}\n\nOutput ONLY the JSON array. Do NOT include any other text, explanation, or conversational elements outside the JSON array.\nThe JSON array must contain exactly 10 objects, each with 'question' and 'answer' keys.\n\nJSON Format Example:\n```json\n[\n  {{\"question\": \"What is X?\", \"answer\": \"Y.\"}},\n  {{\"question\": \"How does A work?\", \"answer\": \"B.\"}}\n]\n```\n[/INST]\n"""

qna_prompt = PromptTemplate(input_variables=["text"], template=qna_prompt_template)
print("Q&A generation prompt defined.")

# 4. Data Acquisition and Chunking Function Definitions
def scrape_web_page(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        content_tags = soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])
        text_content = '\n'.join([tag.get_text() for tag in content_tags])
        return text_content
    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return None

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.create_documents([text])
    return [chunk.page_content for chunk in chunks]
print("Web scraping and text chunking functions defined.")

# 5. Execute scraping and chunking to populate all_document_chunks
example_url_for_chunking = "https://www.3gpp.org/about-3gpp/what-is-3gpp"
print(f"Attempting to scrape content from: {example_url_for_chunking}")

scraped_text_for_chunking = scrape_web_page(example_url_for_chunking)

all_document_chunks = {}

if scraped_text_for_chunking:
    print(f"Successfully scraped text (first 200 chars):\n{scraped_text_for_chunking[:200]}...")
    document_name = "3gpp_about_page"
    print(f"\nProcessing {document_name}...")

    chunks = chunk_text(scraped_text_for_chunking, chunk_size=1000, chunk_overlap=200)
    all_document_chunks[document_name] = chunks
    print(f"Generated {len(chunks)} chunks for {document_name}.")
    if chunks:
        print(f"First chunk of {document_name}:\n{chunks[0][:500]}...")
    else:
        print(f"No chunks generated for {document_name}.")
else:
    print(f"Could not scrape text from {example_url_for_chunking}. Skipping chunking.")

print(f"Total documents (or web pages) processed for chunking: {len(all_document_chunks)}")
print("--- Full initialization process complete. All necessary variables should now be in scope. ---")

--- Starting full initialization process ---
Logged in to Hugging Face.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Language model 'meta-llama/Llama-2-7b-chat-hf' initialized.
Q&A generation prompt defined.
Web scraping and text chunking functions defined.
Attempting to scrape content from: https://www.3gpp.org/about-3gpp/what-is-3gpp
Successfully scraped text (first 200 chars):
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence Cent...

Processing 3gpp_about_page...
Generated 20 chunks for 3gpp_about_page.
First chunk of 3gpp_about_page:
Generations of Mobile Standards



3GPP Portal (3GU)    [email protected]
AboutAbout 3GPP HomeIntroducing 3GPPPartnersMembershipProject Coordination Group (PCG)Achievement AwardsMobile Competence CentreVacancies MCC Task ForcesFunding & FinancesLiaisons StatementsLegal MattersLogo Usage3GPP privacy policyCall for IPRStatement Regarding Competition Law3GPP FAQs
About 3GPP Home
Introducing 3GPP
Partners
Membership

Project C

```markdown
## Action Required: Manual Colab Runtime Restart

Despite multiple attempts to resolve the `ImportError` for `bitsandbytes` by ensuring installation commands are at the top of the cell and explicitly importing the library, the error persists.

This type of issue often requires a **manual runtime restart** in Google Colab to properly load newly installed libraries into the Python environment.

**Please perform the following steps:**
1.  Go to the Colab menu at the top: `Runtime`
2.  Select `Restart runtime`
3.  After the runtime restarts, re-run all cells from the beginning of the notebook.

Once the runtime is restarted and all cells are run, the `bitsandbytes` library should be correctly recognized, and the LLM initialization process should complete successfully.

## Update Langchain Import in LLM Initialization

### Subtask:
Modify the LLM initialization cell to correctly import `HuggingFacePipeline` from `langchain_huggingface.llms` instead of `langchain_community.llms`.


```markdown
## Action Required: Manual Colab Runtime Restart

Despite multiple attempts to resolve the `ImportError` for `bitsandbytes` by ensuring installation commands are at the top of the cell and explicitly importing the library, the error persists.

This type of issue often requires a **manual runtime restart** in Google Colab to properly load newly installed libraries into the Python environment.

**Please perform the following steps:**
1.  Go to the Colab menu at the top: `Runtime`
2.  Select `Restart runtime`
3.  After the runtime restarts, re-run all cells from the beginning of the notebook.

Once the runtime is restarted and all cells are run, the `bitsandbytes` library should be correctly recognized, and the LLM initialization process should complete successfully.
```

## Re-initialize LLM and Prompt

### Subtask:
Execute the updated LLM initialization cell (d2bec2a0) to ensure `llm` and `qna_prompt` are correctly defined with the `langchain-huggingface` pipeline.


**Reasoning**:
Execute the LLM initialization cell to define `llm` and `qna_prompt`.



In [24]:
import json
import re # Import regex module
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface.llms import HuggingFacePipeline # Updated import
import torch
from huggingface_hub import login
from google.colab import userdata

# Get HF_TOKEN from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')
print("HF_TOKEN loaded from Colab secrets.")

# Initialize the Language Model for Llama-2
model_name = "meta-llama/Llama-2-7b-chat-hf" # Using Llama-2-7b-chat-hf

# Configure 4-bit quantization
bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True # Added to resolve ValueError for offloading
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN # Use the HF_TOKEN for gated models
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bitsandbytes_config,
    device_map="auto", # Automatically maps the model to available devices (GPU)
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN # Use the HF_TOKEN for gated models
    # Removed use_flash_attention_2=False as it's not a direct argument for LlamaForCausalLM's __init__
)

# Create the text-generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # device_map="auto" handled by model loading, no need for 'device'
    return_full_text=False, # Crucial: only return generated text, not the prompt
    max_new_tokens=1000, # Increased for Llama-2's capability
    temperature=0.7, # Add a temperature for more diverse outputs
    top_p=0.9 # Top-p sampling
)

llm = HuggingFacePipeline(pipeline=generator)
print(f"\nLanguage model '{model_name}' initialized.")

# Define the prompt template
qna_prompt_template = """[INST] You are an expert in 3GPP telecommunications.\nYour task is to read the following 3GPP specification text and generate exactly 10 question-answer pairs based on the content.\nThe questions should be specific and the answers should be concise and directly derived from the provided text.\nMake sure the questions cover different aspects of the text.\n\nText: {text}\n\nOutput ONLY the JSON array. Do NOT include any other text, explanation, or conversational elements outside the JSON array.\nThe JSON array must contain exactly 10 objects, each with 'question' and 'answer' keys.\n\nJSON Format Example:\n```json\n[\n  {{\"question\": \"What is X?\", \"answer\": \"Y.\"}},\n  {{\"question\": \"How does A work? \", \"answer\": \"B.\"}}\n]\n```\n[/INST]\n"""

# Create the prompt template instance
qna_prompt = PromptTemplate(
    input_variables=["text"],
    template=qna_prompt_template
)
print("Q&A generation prompt defined.")

# Placeholder for 3GPP content. REPLACE THIS WITH YOUR ACTUAL DATA.
# You might load this from a file, scrape a webpage, or have it as a string.
example_3gpp_text = """3GPP Technical Specification 23.501 V16.8.0 (2020-12)\n5G System Architecture\n\n5.1 General Principles\nThe 5G System architecture is defined to support various services and use cases, including enhanced Mobile Broadband (eMBB), Ultra-Reliable Low Latency Communication (URLLC), and massive Machine Type Communication (mMTC). It builds upon the existing 4G LTE architecture with significant enhancements to meet the requirements of new services. Key architectural principles include service-based architecture, network slicing, and control/user plane separation.\n\n5.2 Network Functions\nKey Network Functions (NFs) in the 5G System include: AMF (Access and Mobility Management Function), SMF (Session Management Function), UPF (User Plane Function), UDM (Unified Data Management), AUSF (Authentication Server Function), PCF (Policy Control Function), NEF (Network Exposure Function), and NRF (NF Repository Function).\n\n5.2.1 AMF (Access and Mobility Management Function)\nThe AMF is responsible for Connection Management (CM) and Mobility Management (MM). It handles UE registration, connection setup, reachability management, and mobility between 3GPP access and non-3GPP access. The AMF also performs authentication and authorization with the AUSF and UDM.\n\n5.2.2 SMF (Session Management Function)\nThe SMF is responsible for session management, including session establishment, modification, and release. It selects and controls the UPF, allocates IP addresses, and manages QoS flows.\n\n5.2.3 UPF (User Plane Function)\nThe UPF is the core component for the user plane data handling. It performs packet routing and forwarding, inter-system mobility, and acts as an anchor point for session continuity.\n"""

# Use the LLM to generate Q&A pairs
print("Generating Q&A pairs...")

# Format the prompt with the text
formatted_prompt = qna_prompt.format(text=example_3gpp_text)

# Generate the response using the invoke method
raw_response = llm.invoke(formatted_prompt)

# --- Start of Modified Parsing Logic ---\nqna_pairs = []\n
# Try to extract the JSON block using markdown code block syntax if the model adheres to it\njson_block_match = re.search(r'```json\\s*(\\[.*?\\])\\s*```', raw_response, re.DOTALL)\n\nif json_block_match:\n    json_string = json_block_match.group(1)\n    try:\n        qna_pairs = json.loads(json_string)\n        # Validate that each item in the list is a dict with 'question' and 'answer'\n        if not all(isinstance(item, dict) and 'question' in item and 'answer' in item for item in qna_pairs):\n            print(\"Warning: JSON block was found but its structure is not as expected. Attempting regex fallback.\")\n            qna_pairs = [] # Reset to try regex fallback\n    except json.JSONDecodeError as e:\n        print(f\"Warning: Failed to parse extracted JSON block with json.loads: {e}. Attempting regex fallback.\")\n        qna_pairs = [] # Reset to try regex fallback\nelse:\n    print(\"No `json` markdown block found. Attempting direct JSON parsing or regex fallback.\")\n\nif not qna_pairs:\n    # Fallback to the original regex approach if direct json.loads or markdown block extraction fails\n    # Regex to find all occurrences of {\\"question\\": \\"...\\", \\"answer\\": \\"...\\"}\n    # It's made more robust by handling potential escaped quotes within the string values\n    json_pattern = re.compile(r'{\\s*\\"question\\"\\s*:\\s*\\"(.*?)(?<!\\\\)\\"\\s*,\\s*\\"answer\\"\\s*:\\s*\\"(.*?)(?<!\\\\)\\"\\s*}', re.DOTALL)\n    matches = json_pattern.findall(raw_response)\n\n    if matches:\n        for q_text, a_text in matches:\n            # Unescape quotes that might have been escaped by the model within the strings\n            question = q_text.replace('\\\\"', '\"')\n            answer = a_text.replace('\\\\"', '\"')\n            qna_pairs.append({\"question\": question, \"answer\": answer})\n
if qna_pairs:
    print(f"\nSuccessfully extracted {len(qna_pairs)} potential Q&A pairs:")
    for i, qa in enumerate(qna_pairs):
        print(f"Q{i+1}: {qa['question']}")
        print(f"A{i+1}: {qa['answer']}\n")
else:
    print("\nNo valid Q&A pairs could be extracted from the model output. Raw response:\n")
    print(raw_response)
# --- End of Modified Parsing Logic ---

print("\nStep 4 Complete: Q&A generation attempt finished. Review the output for quality and completeness.")

HF_TOKEN loaded from Colab secrets.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Language model 'meta-llama/Llama-2-7b-chat-hf' initialized.
Q&A generation prompt defined.
Generating Q&A pairs...

Successfully extracted 10 potential Q&A pairs:
Q1: What is the purpose of the AMF?
A1: The AMF is responsible for Connection Management (CM) and Mobility Management (MM).

Q2: What is the role of the SMF?
A2: The SMF is responsible for session management, including session establishment, modification, and release.

Q3: What is the function of the UPF?
A3: The UPF is the core component for the user plane data handling and performs packet routing and forwarding, inter-system mobility, and acts as an anchor point for session continuity.

Q4: What are the key Network Functions in the 5G System?
A4: Key Network Functions (NFs) in the 5G System include: AMF, SMF, UPF, UDM, AUSF, PCF, NEF, and NRF.

Q5: What is the purpose of the AUSF?
A5: The AUSF performs authentication and authorization with the UDM.

Q6: What is the function of the UDM?
A6: The UDM manages the authenticatio